# Capítulos 7-10: Funções, Métodos Numéricos e Projetos Integrados

**Disciplina:** FSC1189 - Algoritmo e Programação

Exercícios avançados com métodos numéricos e simulações físicas.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from scipy.optimize import fsolve

## Exercício 7.1: Módulo de Funções Físicas

**Enunciado:**

Crie um módulo com funções para calcular energias em diferentes contextos.

In [ ]:
class FisicaBasica:
    """Módulo com funções básicas de Física."""
    
    g = 9.81  # aceleração da gravidade (m/s²)
    
    @staticmethod
    def energia_cinetica(massa, velocidade):
        """
        Calcula energia cinética: Ec = (1/2)mv²
        
        Args:
            massa: em kg
            velocidade: em m/s
        
        Returns:
            float: energia em joules
        """
        return 0.5 * massa * velocidade**2
    
    @staticmethod
    def energia_potencial(massa, altura, g=9.81):
        """
        Calcula energia potencial gravitacional: Ep = mgh
        
        Args:
            massa: em kg
            altura: em metros
            g: aceleração da gravidade (padrão 9.81 m/s²)
        
        Returns:
            float: energia em joules
        """
        return massa * g * altura
    
    @staticmethod
    def energia_mecanica(massa, velocidade, altura, g=9.81):
        """
        Calcula energia mecânica total: Em = Ec + Ep
        """
        Ec = FisicaBasica.energia_cinetica(massa, velocidade)
        Ep = FisicaBasica.energia_potencial(massa, altura, g)
        return Ec + Ep
    
    @staticmethod
    def trabalho(forca, distancia, cosseno_angulo=1):
        """
        Calcula trabalho: W = F·d·cos(θ)
        
        Args:
            forca: força em newtons
            distancia: em metros
            cosseno_angulo: cos(θ), onde θ é ângulo entre F e d
        
        Returns:
            float: trabalho em joules
        """
        return forca * distancia * cosseno_angulo

# Teste
m = 5.0  # kg
v = 10.0  # m/s
h = 2.0  # m

Ec = FisicaBasica.energia_cinetica(m, v)
Ep = FisicaBasica.energia_potencial(m, h)
Em = FisicaBasica.energia_mecanica(m, v, h)

print("=== ENERGIAS DE UM OBJETO ===")
print(f"Massa: {m} kg")
print(f"Velocidade: {v} m/s")
print(f"Altura: {h} m")
print()
print(f"Energia cinética: {Ec:.2f} J")
print(f"Energia potencial: {Ep:.2f} J")
print(f"Energia mecânica: {Em:.2f} J")

---

## Exercício 8.1: Método da Bisseção

**Enunciado:**

Encontre a raiz da função $f(x) = x^3 - 2$ no intervalo [1, 2] usando o método da bisseção.

In [ ]:
def f_exemplo(x):
    """Função de teste: f(x) = x³ - 2"""
    return x**3 - 2

def metodo_bisseccao(f, a, b, tolerancia=1e-6, max_iter=100):
    """
    Encontra raiz usando o método da bisseção.
    
    Args:
        f: função
        a, b: intervalo [a, b]
        tolerancia: precisão desejada
        max_iter: máximo de iterações
    
    Returns:
        tuple: (raiz, iteracoes, erros)
    """
    if f(a) * f(b) >= 0:
        raise ValueError("f(a) e f(b) devem ter sinais opostos")
    
    erros = []
    
    for i in range(max_iter):
        c = (a + b) / 2
        fc = f(c)
        erro = abs(fc)
        erros.append(erro)
        
        if erro < tolerancia:
            return c, i + 1, erros
        
        if f(a) * fc < 0:
            b = c
        else:
            a = c
    
    return c, max_iter, erros

# Resolver
raiz, iteracoes, erros = metodo_bisseccao(f_exemplo, 1, 2)
raiz_real = 2**(1/3)

print("=== MÉTODO DA BISSEÇÃO ===")
print(f"Função: f(x) = x³ - 2")
print(f"Intervalo: [1, 2]")
print()
print(f"Raiz encontrada: {raiz:.10f}")
print(f"Raiz real: {raiz_real:.10f}")
print(f"Diferença: {abs(raiz - raiz_real):.2e}")
print(f"Número de iterações: {iteracoes}")
print(f"f(raiz) = {f_exemplo(raiz):.2e}")

### Convergência do Método

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Gráfico da função
x = np.linspace(0.5, 2.5, 200)
y = f_exemplo(x)
ax1.plot(x, y, 'b-', linewidth=2, label='f(x) = x³ - 2')
ax1.axhline(0, color='k', linestyle='-', linewidth=0.5)
ax1.plot(raiz, 0, 'ro', markersize=10, label=f'Raiz = {raiz:.6f}')
ax1.set_xlabel('x', fontsize=11)
ax1.set_ylabel('f(x)', fontsize=11)
ax1.set_title('Função e Raiz Encontrada', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.legend(fontsize=10)

# Convergência dos erros
iteracoes_plot = list(range(1, len(erros) + 1))
ax2.semilogy(iteracoes_plot, erros, 'g-o', linewidth=2, markersize=4)
ax2.set_xlabel('Iteração', fontsize=11)
ax2.set_ylabel('|f(c)| (escala log)', fontsize=11)
ax2.set_title('Convergência do Método', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

---

## Exercício 8.2: Diferenciação Numérica

**Enunciado:**

Calcule numericamente a derivada de $f(x) = \sin(x)$ usando diferenças finitas e compare com o resultado analítico (que é $\cos(x)$).

In [ ]:
def f_sen(x):
    return np.sin(x)

def f_sen_derivada_real(x):
    return np.cos(x)

def derivada_numerica_central(f, x, h=1e-5):
    """
    Calcula derivada usando diferença central:
    f'(x) ≈ [f(x+h) - f(x-h)] / (2h)
    """
    return (f(x + h) - f(x - h)) / (2 * h)

def derivada_numerica_progressiva(f, x, h=1e-5):
    """
    Calcula derivada usando diferença progressiva:
    f'(x) ≈ [f(x+h) - f(x)] / h
    """
    return (f(x + h) - f(x)) / h

# Teste para x = π/4
x_teste = np.pi / 4

der_real = f_sen_derivada_real(x_teste)
der_central = derivada_numerica_central(f_sen, x_teste)
der_progressiva = derivada_numerica_progressiva(f_sen, x_teste)

print("=== DIFERENCIAÇÃO NUMÉRICA ===")
print(f"Função: f(x) = sin(x)")
print(f"Ponto: x = π/4 ≈ {x_teste:.6f}")
print()
print(f"Derivada real (cos(π/4)): {der_real:.10f}")
print(f"Derivada numérica (diferença central): {der_central:.10f}")
print(f"Derivada numérica (diferença progressiva): {der_progressiva:.10f}")
print()
print(f"Erro (central): {abs(der_central - der_real):.2e}")
print(f"Erro (progressiva): {abs(der_progressiva - der_real):.2e}")

### Análise de Precisão

In [ ]:
# Variar h e analisar erro
h_valores = np.logspace(-8, -2, 50)
erros_central = []
erros_progressiva = []

for h in h_valores:
    der_c = derivada_numerica_central(f_sen, x_teste, h)
    der_p = derivada_numerica_progressiva(f_sen, x_teste, h)
    
    erros_central.append(abs(der_c - der_real))
    erros_progressiva.append(abs(der_p - der_real))

plt.figure(figsize=(10, 6))
plt.loglog(h_valores, erros_central, 'b-o', label='Diferença Central', markersize=4)
plt.loglog(h_valores, erros_progressiva, 'r-s', label='Diferença Progressiva', markersize=4)
plt.xlabel('Tamanho do passo (h)', fontsize=11)
plt.ylabel('Erro Absoluto (escala log)', fontsize=11)
plt.title('Análise de Precisão - Diferenciação Numérica', fontsize=12, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

---

## Exercício 8.3: Integração Numérica - Regra do Trapézio

**Enunciado:**

Calcule a integral $\int_0^{\pi} \sin(x) dx$ usando a regra do trapézio.

Valor teórico: $\int_0^{\pi} \sin(x) dx = 2$

In [ ]:
def regra_trapezoidal(f, a, b, n):
    """
    Calcula integral usando regra dos trapézios.
    ∫_a^b f(x)dx ≈ (h/2)[f(x₀) + 2f(x₁) + 2f(x₂) + ... + f(xₙ)]
    """
    h = (b - a) / n
    x = np.linspace(a, b, n + 1)
    y = f(x)
    
    # Fórmula dos trapézios
    integral = h * (y[0] + 2*np.sum(y[1:-1]) + y[-1]) / 2
    
    return integral, x, y

# Calcular para diferentes números de subintervalos
a = 0
b = np.pi
valor_teorico = 2.0

n_valores = [10, 50, 100, 500, 1000]

print("=== INTEGRAÇÃO NUMÉRICA - REGRA DO TRAPÉZIO ===")
print(f"Integral: ∫₀^π sin(x)dx")
print(f"Valor teórico: {valor_teorico}")
print()
print("n     | Valor Aproximado | Erro Absoluto | Erro (%)")
print("-" * 55)

erros = []
for n in n_valores:
    integral, _, _ = regra_trapezoidal(np.sin, a, b, n)
    erro = abs(integral - valor_teorico)
    erro_percent = (erro / valor_teorico) * 100
    
    erros.append(erro)
    print(f"{n:<5} | {integral:16.10f} | {erro:13.2e} | {erro_percent:7.4f}%")

### Visualização

In [ ]:
# Plotar aproximação com n=20
n = 20
integral, x_trap, y_trap = regra_trapezoidal(np.sin, 0, np.pi, n)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Gráfico da aproximação
x_continuo = np.linspace(0, np.pi, 200)
y_continuo = np.sin(x_continuo)

ax1.plot(x_continuo, y_continuo, 'b-', linewidth=2, label='sin(x)')
ax1.fill_between(x_continuo, y_continuo, alpha=0.2, color='blue')
ax1.plot(x_trap, y_trap, 'ro-', markersize=5, label=f'Aproximação (n={n})')

# Trapézios
for i in range(len(x_trap) - 1):
    ax1.fill_between([x_trap[i], x_trap[i+1]], 
                     [y_trap[i], y_trap[i+1]], 
                     alpha=0.1, color='red')

ax1.set_xlabel('x', fontsize=11)
ax1.set_ylabel('sin(x)', fontsize=11)
ax1.set_title(f'Aproximação com Trapézios (n={n})', fontsize=12, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Convergência do erro
ax2.loglog(n_valores, erros, 'go-', linewidth=2, markersize=8)
ax2.set_xlabel('Número de subintervalos (n)', fontsize=11)
ax2.set_ylabel('Erro Absoluto (escala log)', fontsize=11)
ax2.set_title('Convergência da Regra do Trapézio', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

---

## Exercício 10.1: Projeto - Análise de Dados de Pêndulo Simples

**Cenário:**

Dados experimentais de período de um pêndulo em função do comprimento.
Objetivo: Estimar $g$ a partir da relação $T = 2\pi\sqrt{L/g}$

In [ ]:
# Dados experimentais (simulados com ruído)
np.random.seed(42)
L_teorico = np.array([0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0])  # metros
g_real = 9.81  # m/s²

# Calcular períodos teóricos
T_teorico = 2 * np.pi * np.sqrt(L_teorico / g_real)

# Adicionar ruído experimental (1%)
ruido = 0.01 * T_teorico * np.random.randn(len(T_teorico))
T_medido = T_teorico + ruido

# Criar tabela
df_pendulo = pd.DataFrame({
    'Comprimento L (m)': L_teorico,
    'Período T teórico (s)': T_teorico,
    'Período T medido (s)': T_medido,
    'Erro (%)': (np.abs(T_medido - T_teorico) / T_teorico) * 100
})

print("=== DADOS DE PÊNDULO SIMPLES ===")
print(df_pendulo.to_string(index=False))

### Análise de Dados

In [ ]:
# Usar relação: T² = (4π²/g)L para ajustar uma reta
# Se plotar T² vs L, a inclinação = 4π²/g

T_squared = T_medido**2

# Regressão linear
# T² = aL + b, onde a = 4π²/g
coef = np.polyfit(L_teorico, T_squared, 1)
a = coef[0]
b = coef[1]

g_estimado = 4 * np.pi**2 / a

print("\n=== ANÁLISE DE REGRESSÃO LINEAR ===")
print(f"Ajuste: T² = {a:.6f}·L + {b:.6f}")
print(f"Coeficiente angular (a): {a:.6f}")
print(f"Coeficiente linear (b): {b:.6f}")
print()
print(f"g estimado: {g_estimado:.3f} m/s²")
print(f"g real: {g_real:.3f} m/s²")
print(f"Erro: {abs(g_estimado - g_real):.4f} m/s² ({(abs(g_estimado - g_real)/g_real)*100:.2f}%)")

### Visualização

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Gráfico T vs L
ax1.scatter(L_teorico, T_medido, s=80, alpha=0.7, label='Dados medidos', color='blue')
ax1.plot(L_teorico, T_teorico, 'r--', linewidth=2, label='Valor teórico')
ax1.set_xlabel('Comprimento L (m)', fontsize=11)
ax1.set_ylabel('Período T (s)', fontsize=11)
ax1.set_title('Período vs Comprimento', fontsize=12, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Gráfico T² vs L (regressão linear)
L_fit = np.linspace(L_teorico.min(), L_teorico.max(), 100)
T2_fit = a * L_fit + b

ax2.scatter(L_teorico, T_squared, s=80, alpha=0.7, label='Dados medidos', color='green')
ax2.plot(L_fit, T2_fit, 'b-', linewidth=2, label=f'Ajuste linear')
ax2.set_xlabel('Comprimento L (m)', fontsize=11)
ax2.set_ylabel('Período² T² (s²)', fontsize=11)
ax2.set_title('T² vs L - Regressão Linear', fontsize=12, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## Exercício 10.2: Projeto - Simulação de Queda Livre com Arrasto

**Cenário:**

Simular a queda de um objeto considerando resistência do ar.

In [ ]:
def queda_livre_com_arrasto(condicoes_iniciais, parametros, t_total):
    """
    Simula queda livre com arrasto quadrático.
    
    dv/dt = g - (b/m)v|v|
    """
    def derivada(y, t, g, b, m):
        v = y[0]
        aceleracao = g - (b / m) * v * abs(v)
        return [aceleracao]
    
    g = parametros['g']
    b = parametros['b']  # coeficiente de arrasto
    m = parametros['m']  # massa
    
    t = np.linspace(0, t_total, 500)
    solucao = odeint(derivada, condicoes_iniciais, t, args=(g, b, m))
    
    return t, solucao[:, 0]

# Parâmetros
parametros = {
    'g': 9.81,      # aceleração da gravidade
    'b': 0.1,       # coeficiente de arrasto
    'm': 1.0        # massa em kg
}

# Velocidade terminal
v_terminal = np.sqrt(parametros['g'] * parametros['m'] / parametros['b'])

# Simular
t, v = queda_livre_com_arrasto([0], parametros, 10)

# Também simular sem arrasto para comparação
parametros_sem_arrasto = parametros.copy()
parametros_sem_arrasto['b'] = 0.001  # arrasto negligenciável
t2, v_sem_arrasto = queda_livre_com_arrasto([0], parametros_sem_arrasto, 10)

print("=== SIMULAÇÃO: QUEDA COM ARRASTO ===")
print(f"Massa: {parametros['m']} kg")
print(f"Coeficiente de arrasto: {parametros['b']}")
print(f"Velocidade terminal: {v_terminal:.2f} m/s")

### Visualização

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(t, v, 'b-', linewidth=2, label=f'Com arrasto (b={parametros["b"]})')
plt.plot(t2, v_sem_arrasto, 'r--', linewidth=2, label='Sem arrasto')
plt.axhline(v_terminal, color='g', linestyle=':', linewidth=2, label=f'Velocidade terminal = {v_terminal:.2f} m/s')

plt.xlabel('Tempo (s)', fontsize=11)
plt.ylabel('Velocidade (m/s)', fontsize=11)
plt.title('Queda Livre: Com e Sem Arrasto', fontsize=12, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## Exercício Proposto 5: Lei de Coulomb Numérica

**Desafio:**

Implemente o cálculo da força eletrostática entre múltiplas cargas em um espaço 2D.

$$F = k \frac{|q_1 q_2|}{r^2}$$

Visualize as forças com vetores.

In [ ]:
# Sua solução aqui
# Dica: crie funções para calcular distância e força entre pares de cargas

---

## Exercício Proposto 6: Oscilador Harmônico Amortecido

**Desafio:**

Simule o movimento de um oscilador harmônico com amortecimento:

$$\frac{d^2x}{dt^2} + 2\gamma\frac{dx}{dt} + \omega_0^2 x = 0$$

Teste para diferentes graus de amortecimento (subamortecido, criticamente amortecido, superamortecido).

In [ ]:
# Sua solução aqui
# Dica: use odeint para resolver a EDO de segunda ordem